# End-to-End Agentic Pick-Up Resolution

This notebook demonstrates the full agentic loop for resolving an **underspecified** `PickUpAction` KRROOD `Match` into a fully executable PyCRAM action instance.

## Scenario
A developer provides only the _intent_ — `object_designator=milk_body` — and leaves everything else open:
```
underspecified(PickUpAction)(object_designator=milk_body)
                                ↓
   arm                 = ???   ← must be decided by reachability analysis
   grasp_description   = ???   ← approach direction, alignment, manipulator
```

The `AgenticLLMBackend` receives this `Match`, detects the free/gap parameters via `snapshot_match`, and delegates to its three specialist sub-agents:

| Sub-agent | Role in this task |
|---|---|
| **SceneQueryAgent** | Locate the milk body, get its pose, dimensions, and orientation |
| **KinematicsAgent** | Check left/right arm reachability; return valid grasp poses |
| **PlanningAgent** | Look up `PickUpAction` schema; confirm required parameter types |

The Orchestrator synthesises the findings into a JSON parameter set, which the backend hydrates into a live `PickUpAction` instance.

## 0. Environment Setup

In [1]:
import os


# ROS / ament package paths — must be set before SDT/PyCRAM imports
_IAI_PR2_INSTALL = '/home/malineni/ros_packages/iai_pr2_install'
_IAI_PR2_SRC     = '/home/malineni/ros_packages/iai_pr2/iai_pr2_description'
os.environ['AMENT_PREFIX_PATH'] = _IAI_PR2_INSTALL + ':' + os.environ.get('AMENT_PREFIX_PATH', '')
os.environ['ROS_PACKAGE_PATH']  = _IAI_PR2_SRC     + ':' + os.environ.get('ROS_PACKAGE_PATH', '')

# API key — set here or export in your shell before launching Jupyter
# os.environ['ANTHROPIC_API_KEY'] = 'REPLACE_WITH_OPENAI_API_KEY...'
os.environ["OPENAI_API_KEY"] = "REPLACE_WITH_OPENAI_API_KEY"

print('Environment configured.')

Environment configured.


## 1. Load World and Initialise the Agentic Backend

We load a ready-made PR2 world (table + milk carton + cereal box), register it as the active world so all SDT tools can query it, then wire up the LLM and the `AgenticLLMBackend`.

In [2]:
from langchain_openai import ChatOpenAI          # swap for ChatAnthropic if preferred
from agentic_llmr.integrations.world_manager import set_active_world
from agentic_llmr.backend import AgenticLLMBackend
from uniworld import load_pr2_simple_world

# Load world: PR2 + table @ 1.1 m + milk @ (0.9, 0.0, 0.85) + cereal @ (0.9, 0.2, 0.85)
print('Loading PR2 simple world...')
world, robot_view, context = load_pr2_simple_world()
set_active_world(world, robot_view)
print(f'World ready — {len(list(world.bodies))} bodies, '
      f'{len(list(world.semantic_annotations))} semantic annotations.')

# Initialise LLM  (gpt-4o recommended for multi-step tool chaining)
llm = ChatOpenAI(model='gpt-4o', temperature=0.0)

# AgenticLLMBackend wraps the ReActAgent orchestrator
backend = AgenticLLMBackend(llm=llm)
print('AgenticLLMBackend ready.')

Loading PR2 simple world...


Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]


World ready — 114 bodies, 16 semantic annotations.
AgenticLLMBackend ready.


## 2. Build the Underspecified Match

`underspecified(PickUpAction)` creates a KRROOD `Match` that is intended for generative backends — unlike `match_variable` it does **not** search an existing domain.

We bind only `object_designator` to the live milk body retrieved from the world. Every other required field (`arm`, `grasp_description`) is intentionally omitted.

In [3]:
from krrood.entity_query_language.factories import underspecified
from pycram.robot_plans.actions.core.pick_up import PickUpAction

# Retrieve the milk body from the live world
milk_body = world.get_body_by_name('milk.stl')
print(f'Milk body      : {milk_body}')
print(f'Milk body type : {type(milk_body).__name__}')

# Create a Match with only object_designator bound
# arm and grasp_description are left entirely unspecified
pickup_match = underspecified(PickUpAction)(object_designator=milk_body)
print(f'\nMatch created  : {pickup_match}')
print(f'Bound kwargs   : {list(pickup_match.kwargs)}')

Milk body      : Body(name=PrefixedName('None/milk.stl'), id=UUID('113cac16-b08f-4534-ace1-0a0a4d56d08e'), index=112)
Milk body type : Body

Match created  : Match(PickUpAction)
Bound kwargs   : ['object_designator']


### Inspect What the Backend Sees

`snapshot_match` is the first thing `AgenticLLMBackend._evaluate()` calls internally. Running it here lets us see exactly which parameters are fixed vs free before the agent starts.

In [4]:
from agentic_llmr.resolution.action_match import snapshot_match
from agentic_llmr.resolution.schema_docs import build_action_documentation

template = snapshot_match(pickup_match)

print(f'Action class   : {template.action_name}')

print(f'\nFixed bindings (already resolved):')
for name, val in template.fixed_bindings.items():
    # Show display name for SDT Symbol objects
    try:
        from krrood.symbol_graph.symbol_graph import Symbol
        from agentic_llmr.resolution.scene import symbol_display_name
        rendered = f'"{symbol_display_name(val)}" (body)' if isinstance(val, Symbol) else repr(val)
    except Exception:
        rendered = repr(val)
    print(f'  {name:30s} = {rendered}')

print(f'\nFree parameters (agent must resolve):')
for p in template.free_parameters:
    import dataclasses
    from enum import Enum
    ft = p.field_type
    gap = ' [gap — not in Match]' if p._variable is None else ''
    if isinstance(ft, type) and issubclass(ft, Enum):
        members = ' | '.join(e.name for e in ft)
        print(f'  {p.prompt_name:30s} : Enum({members}){gap}')
    elif isinstance(ft, type) and dataclasses.is_dataclass(ft):
        print(f'  {p.prompt_name:30s} : {ft.__name__} dict{gap} — see schema below')
    else:
        type_label = getattr(ft, '__name__', str(ft))
        print(f'  {p.prompt_name:30s} : {type_label}{gap}')

print(f'\nOptional defaults (agent may override):')
for name, val in template.optional_default_bindings.items():
    print(f'  {name:30s} = {val!r}')

print(f'\n--- Full Action Schema (what the backend sends to the agent) ---')
print(build_action_documentation(template.action_type))
print('\nNote: Manipulator is auto-injected from arm — do NOT include it in JSON.')

Action class   : PickUpAction

Fixed bindings (already resolved):
  object_designator              = "milk.stl" (body)

Free parameters (agent must resolve):
  arm                            : Enum(LEFT | RIGHT | BOTH) [gap — not in Match]
  grasp_description              : GraspDescription dict [gap — not in Match] — see schema below

Optional defaults (agent may override):

--- Full Action Schema (what the backend sends to the agent) ---
Action Class: PickUpAction
Description: Let the robot pick up an object.

Required Free Parameters:
  - object_designator (Dict[Body]) — Object designator_description describing the object that should be picked up. Required keys:
  - arm (Enum: LEFT | RIGHT | BOTH) — The arm that should be used for pick up
  - grasp_description (Dict[GraspDescription]) — The GraspDescription that should be used for picking up the object. Required keys:
      - approach_direction (Enum: FRONT | BACK | RIGHT | LEFT) — The direction from which the body should be grasped

## 3. Submit the Match to the AgenticLLMBackend

Calling `backend._evaluate(pickup_match)` triggers the full agentic loop:

1. `snapshot_match` snapshots bound/free params into an `ActionTemplate`.
2. The template context is formatted and sent to the **Orchestrator**.
3. The Orchestrator calls sub-agents in order:
   - `query_action_schema` → PlanningAgent discovers `PickUpAction` parameter types
   - `query_scene_perception` → SceneQueryAgent fetches milk pose, dimensions, orientation
   - `query_kinematics` → KinematicsAgent checks left/right arm reachability and returns grasp poses
4. The Orchestrator outputs a JSON parameter block.
5. The backend hydrates the JSON into a live `PickUpAction` via KRROOD-native construction.

> The full reasoning trace is printed below as the agent streams its responses.

In [5]:
# Trigger the full agentic resolution pipeline
results = list(backend._evaluate(pickup_match))

print(f'\n{'='*60}')
print(f'Resolution complete — {len(results)} instance(s) returned.')

[AgenticLLMBackend] Routing input to ReActAgent...
[AgenticLLMBackend] Action: PickUpAction | free: ['arm', 'grasp_description'] | bound: ['object_designator']

--- [ORCHESTRATOR STARTED] ---
[Orchestrator]:
Instruction: Resolve the free parameters for the provided action.
Context: Action Class: PickUpAction

Already-bound parameters (do NOT re-resolve these):
  object_designator = "milk.stl"  (body name in active world)

Free parameters that must be resolved from the scene:
  arm  (Enum: LEFT | RIGHT | BOTH)
  grasp_description  (GraspDescription dict — see schema below for required keys)

--- Full Parameter Schema (use this to build your JSON output) ---
Action Class: PickUpAction
Description: Let the robot pick up an object.

Required Free Parameters:
  - object_designator (Dict[Body]) — Object designator_description describing the object that should be picked up. Required keys:
  - arm (Enum: LEFT | RIGHT | BOTH) — The arm that should be used for pick up
  - grasp_description (Dict

## 4. Inspect the Resolved Action

The backend returns a fully hydrated `PickUpAction` instance with all fields populated. Inspect each field to verify the agent's decisions.

In [6]:
if not results:
    print('No result returned — check the agent trace above for errors.')
else:
    action = results[0]
    print(f'Type                : {type(action).__name__}')
    print(f'object_designator   : {action.object_designator}')
    print(f'arm                 : {action.arm}')

    gd = action.grasp_description
    if gd is not None:
        print(f'grasp_description   :')
        print(f'  approach_direction  : {gd.approach_direction}')
        print(f'  vertical_alignment  : {gd.vertical_alignment}')
        print(f'  manipulator         : {gd.manipulator}')
        print(f'  rotate_gripper      : {gd.rotate_gripper}')
        print(f'  manipulation_offset : {gd.manipulation_offset}')
    else:
        print('grasp_description   : None (check agent trace)')

Type                : PickUpAction
object_designator   : Body(name=PrefixedName('None/milk.stl'), id=UUID('113cac16-b08f-4534-ace1-0a0a4d56d08e'), index=112)
arm                 : LEFT
grasp_description   :
  approach_direction  : ApproachDirection.FRONT
  vertical_alignment  : VerticalAlignment.NoAlignment
  manipulator         : ParallelGripper(name=PrefixedName('pr2/left_gripper'), id=UUID('8738092c-3a1b-43cb-819e-71f41a6c43e2'), root=Body(name=PrefixedName('pr2/l_gripper_palm_link'), id=UUID('438adac8-ab12-4e59-acaf-6678edb01498'), index=84), _robot=PR2(neck=Neck(name=PrefixedName('pr2/neck'), id=UUID('d3958a37-f18b-4f1d-9154-88bb08278af4'), root=Body(name=PrefixedName('pr2/head_pan_link'), id=UUID('c91cac69-6c17-4da4-b8ff-44623bbf590e'), index=21), _robot=..., joint_states=[], tip=Body(name=PrefixedName('pr2/head_tilt_link'), id=UUID('029c3d19-5b7b-412c-87e7-9c0d2c9a0e7c'), index=22), manipulator=None, sensors=[Camera(name=PrefixedName('pr2/wide_stereo_optical_frame'), id=UUID('e3

## 5. Execute (Optional)

With a MuJoCo simulation running, the resolved action can be executed directly. See `pycram_mujoco_demo.ipynb` for the full simulation setup.

```python
from pycram.motion_executor import simulated_robot
with simulated_robot:
    action.perform()
```